# EX-FEVER: recursive multi-hop fact verification

The notebook evaluates three procedures with the same retriever and verdict head:

- `retrieve`: retrieve once using the complete claim.
- `decompose`: generate subquestions and answer each one; splitting is disabled.
- `split`: use the same decomposition and adaptively split an unanswered subquestion.

The active setting is three-way EX-FEVER with hybrid BM25+BGE retrieval and `gpt-4o-mini`.


## 0. What this notebook uses from outside

Data and infrastructure only. Every part of the **method** — prompts and functions — is defined
in this notebook: the question tree in section 9, the claim tree in section 10.

In [ ]:
# ============================================================================
# EXTERNAL FILES THIS NOTEBOOK USES -- data and infrastructure only.
# Every part of the METHOD is defined in this notebook: the question tree in section 9, the
# claim tree in section 10. Nothing below is imported from the project's own modules.
# ============================================================================
#
#  Environment variable OPENAI_API_KEY (or a local .env loaded by the user)
#      API key for the reader. Nothing else leaves the machine.
#
#  exfever/build_exfever.py               DATASET PREP, run once
#      Downloads EX-FEVER's own files from github.com/dependentsign/EX-FEVER and converts them
#      into the claim schema this notebook reads. Writes the two derived files below.
#
#  exfever/raw/mini_test.csv              DATASET  1,000 claims (the authors' LLM-eval split)
#  exfever/raw/wiki_db.db                 DATASET  the 50,231-document corpus they ship
#  exfever/exfever_mini_test_all.json     derived claims
#  exfever/exfever_corpus.json            derived corpus, 49,197 documents
#
#  exfever/local_retrievers.py            RETRIEVERS -- local, free, no API
#      BM25Retriever  bm25s over the corpus, CPU, ~12 s to build
#      BGERetriever   BAAI/bge-large-en-v1.5 on the GPU, ~45 min once, then cached
#      load_corpus()  reads exfever_corpus.json, drops empty and duplicate texts
#
#  exfever/bm25_index/                    prebuilt index    ~40 MB
#  exfever/emb_bge_large_en_v15.npy       prebuilt vectors  202 MB, 49,197 x 1024
#
# NOT used: sec13_musique.py, exfever_prompts.py, run_exfever.py. What those contain of the
# method is inlined below, so this notebook is self-contained and cannot silently drift from it.
print("external: dataset + retrievers only -- the method is defined in this notebook")

## 1. Install

In [ ]:
!pip install -q bm25s sentence-transformers openai python-dotenv pandas pyarrow tqdm

## 2. API key — reader only; embeddings never leave the machine

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # OPENAI_API_KEY and/or OPENROUTER_API_KEY

In [ ]:
import os, getpass
if not (os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY")):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")
print("OpenRouter key set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OpenAI key set:    ", bool(os.environ.get("OPENAI_API_KEY")))

## 3. Imports

In [ ]:
import collections, json, io, os, random, re, string, sys, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from math import comb

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from IPython.display import display   # so the preview cells also work if run as a plain script

PROJ = os.path.abspath(os.getcwd())
if os.path.basename(PROJ).lower() != "notebooks" and os.path.isdir(os.path.join(PROJ, "notebooks")):
    PROJ = os.path.join(PROJ, "notebooks")
sys.path.insert(0, PROJ)
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

## 4. Config

In [ ]:
# ---------------- Configuration ----------------
N_SAMPLES = 1000
SEED = 500
TOP_K = 5
SOLVE_K = 3
MAX_SPLIT_DEPTH = 2
EVIDENCE_CAP = 3 * TOP_K

RETRIEVER_NAME = "hybrid"
READER_MODEL = "gpt-4o-mini"
REASONING_EFFORT = None
ARMS = ["split"]
WORKERS = 12
CALL_CAP = 15000

HERE = os.path.join(PROJ, "exfever")
sys.path.insert(0, HERE)
print(f"{N_SAMPLES} claims | {RETRIEVER_NAME} | {READER_MODEL} | 3-way")


In [ ]:
# ---------------- Reader ----------------

_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("reader endpoint: OpenAI direct")

_calls = collections.Counter()
_lock = threading.Lock()
_rl_until, _rl_lock = [0.0], threading.Lock()


class QuotaExhausted(RuntimeError):
    pass


def _rl_wait():
    while True:
        with _rl_lock:
            remaining = _rl_until[0] - time.monotonic()

        if remaining <= 0:
            return

        time.sleep(min(remaining, 5.0))


def llm(system, user, max_tokens=64):
    token_parameter = (
        "max_completion_tokens"
        if re.match(r"^(gpt-5|o\d)", READER_MODEL)
        else "max_tokens"
    )

    if REASONING_EFFORT:
        request_options = {
            "reasoning_effort": REASONING_EFFORT,
            "seed": SEED,
        }
        reasoning_reserve = 4096
    else:
        request_options = {
            "temperature": 0.0,
            "seed": SEED,
        }
        reasoning_reserve = 0

    with _lock:
        _calls["n"] += 1
        if _calls["n"] > CALL_CAP:
            raise QuotaExhausted(
                f"self-imposed cap of {CALL_CAP} calls reached"
            )

    budget = max_tokens + reasoning_reserve
    budget_increases = 0
    last_error = None

    for attempt in range(8):
        _rl_wait()

        try:
            response = _client.chat.completions.create(
                model=READER_MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                **request_options,
                **{token_parameter: budget},
            )

            output = (
                response.choices[0].message.content or ""
            ).strip()

            if (
                not output
                and response.choices[0].finish_reason == "length"
                and budget_increases < 3
            ):
                budget_increases += 1
                budget *= 4
                continue

            return output

        except Exception as error:
            message = str(error)

            if any(
                marker in message
                for marker in ("insufficient_quota", "no credits", "402")
            ):
                raise QuotaExhausted(
                    "OpenAI billing error: " + message[:160]
                ) from error

            if "429" in message or "rate limit" in message.lower():
                match = re.search(
                    r"try again in ([\d.]+)\s*(ms|s)\b",
                    message,
                )

                if match:
                    retry_after = float(match.group(1))
                    if match.group(2) == "ms":
                        retry_after /= 1000
                else:
                    retry_after = 0.5

                delay = min(
                    max(retry_after, 0.5) * 1.7**attempt
                    + random.random(),
                    45,
                )

                with _rl_lock:
                    _rl_until[0] = max(
                        _rl_until[0],
                        time.monotonic() + delay,
                    )

                last_error = error
                continue

            last_error = error
            time.sleep(1.5**attempt + random.random())

    raise last_error


print("reader ready:", READER_MODEL)

## 5. Load the claims

`build_exfever.py` turns the authors' CSV into the schema the tree reads. It also prints the
**hidden-hop diagnostic** — how many of a claim's gold documents are named verbatim in the claim.
That number, computed with no model at all, predicted the outcome on every benchmark so far.

In [ ]:
import subprocess

claims_file = os.path.join(HERE, "exfever_mini_test_all.json")
if not os.path.exists(claims_file):
    subprocess.run([sys.executable, os.path.join(HERE, "build_exfever.py"),
                    "--split", "mini_test", "--labels", "3way", "--n", "0",
                    "--out-prefix", "exfever_mini_test_all"], cwd=HERE, check=True)

ALL_CLAIMS = json.load(io.open(claims_file, encoding="utf-8"))
rng = random.Random(SEED)
CLAIMS = sorted(rng.sample(ALL_CLAIMS, N_SAMPLES), key=lambda c: c["id"])
print(f"{len(CLAIMS)} claims of {len(ALL_CLAIMS)} available")
print("labels:", dict(collections.Counter(c["label"] for c in CLAIMS)))


In [ ]:
import local_retrievers as LR

PASSAGES = LR.load_corpus()
print(f"{len(PASSAGES):,} Wikipedia articles loaded")


In [ ]:
if not os.path.isdir(LR.BM25_DIR):
    LR.build_bm25(PASSAGES)
if not os.path.exists(LR.BGE_NPY):
    LR.build_bge(PASSAGES)


class HybridRetriever:
    def __init__(self, passages, rrf_k=60):
        self.passages = passages
        self.rrf_k = rrf_k
        self.retrievers = (LR.BM25Retriever(passages), LR.BGERetriever(passages))

    def retrieve(self, query, k):
        pool = {}
        for retriever in self.retrievers:
            for rank, (passage, _) in enumerate(retriever.retrieve(query, max(k * 4, 20))):
                pool.setdefault(id(passage), [passage, 0.0])
                pool[id(passage)][1] += 1.0 / (self.rrf_k + rank + 1)
        return [(p, score) for p, score in
                sorted(pool.values(), key=lambda x: -x[1])[:k]]


base_retriever = HybridRetriever(PASSAGES)
print("retriever ready: hybrid")


In [ ]:
# DECOMP_SYS = (
#     "You turn a multi-hop CLAIM into the ordered list of questions someone would have to answer "
#     "in order to check it.\n"
#     "A multi-hop claim usually describes an entity instead of naming it - 'the comedian who "
#     "founded Apatow Productions', 'the director of that film' - and its evidence is spread over "
#     "several Wikipedia articles. The early questions resolve those descriptions into names; the "
#     "later ones ask about the properties the claim asserts of them.\n"
#     "Rules:\n"
#     "- One question per line, numbered '1.', '2.', ...\n"
#     "- At most 5 questions.\n"
#     "- When a later question depends on an earlier answer, refer to it with #k "
#     "(e.g. #1 = the answer to question 1). Independent questions do not need a #k reference.\n"
#     "- Resolve every entity the claim describes but does not name before asking about it.\n"
#     "- Ask about EVERY property the claim asserts - dates, nationalities, roles, genres, places. "
#     "A property nobody asks about is never checked, and a false claim is often false precisely "
#     "there.\n"
#     "- Ask, never assert. Write 'Who directed X?', not 'X was directed by Y'. Nothing in the "
#     "claim may be assumed true: that is what is being tested.\n"
#     "- Prefer questions that name the property they are looking for ('What nationality is #1?') "
#     "over yes/no questions, because each question is also used as a search query.\n"
#     "- Each question must be answerable on its own once its #k references are filled in.\n"
#     "Output ONLY the numbered list, nothing else."
# )

# DECOMP_FEWSHOT = (
#     "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
#     "Claim: The founder of Northwind Cartography was born in Harrowmere.\n"
#     "1. Who founded Northwind Cartography?\n"
#     "2. Where was #1 born?\n\n"
#     "Claim: The lead actress of Glass Aviary was born in Dunmarrow.\n"
#     "1. Who is the lead actress of Glass Aviary?\n"
#     "2. Where was #1 born?\n\n"
#     "Claim: The Ashen Ledger was written by Ilvane Sabreth and published in 1998.\n"
#     "1. Who wrote The Ashen Ledger?\n"
#     "2. When was The Ashen Ledger published?\n"
#     "### END EXAMPLES\n\n"
#     "Now the REAL claim. Every question must be about the entities in THIS claim and nothing from "
#     "the examples above.\n"
# )

# ANSWER_SYS = (
#     "You are a knowledge management system. Answer the Question using ONLY the Knowledge Pool. "
#     "Do not use your own world knowledge. You may combine facts if needed. Your answer must be "
#     "a short word or phrase directly supported by the Knowledge Pool. Give only the answer, with "
#     "no explanation. If the Knowledge Pool does not contain the answer, output exactly: unknown."
# )

# GUESS_SYS = (
#     "You are a knowledge management system. Answer the Question using ONLY the Knowledge Pool. "
#     "Do not use your own world knowledge. You may combine facts if needed. Your answer must be "
#     "a short word or phrase from the Knowledge Pool. Even when the pool is a poor match, give "
#     "the closest word or phrase in it; never refuse and never answer 'unknown'. Give only the "
#     "answer, with no explanation."
# )

# BRIDGE_SYS = (
#     "You work ONLY over the retrieved Wikipedia passages, never from your own knowledge.\n"
#     "A reader has already tried and failed to answer the Question from these passages. Do NOT "
#     "answer the question. Your only job is to decide whether there is a stepping stone.\n"
#     "There is a stepping stone when the question is about something the passages refer to but do "
#     "not name, and the passages DO name it while describing something else. Resolving that name "
#     "would make the question answerable. Copy the one sentence that names it.\n"
#     "There is NO stepping stone when the passages simply do not cover what was asked. If the "
#     "passages name the question's own entity but never state the asked fact about it, that is not "
#     "a stepping stone - the corpus does not have the answer, and saying so is the correct and "
#     "useful outcome. Do not reach for a loosely related sentence to avoid answering NONE.\n"
#     "Before you answer, ask yourself: once this name is resolved, would the passages then answer "
#     "the question? If not, output NONE.\n"
#     "Output EXACTLY ONE line:\n"
#     "  BRIDGE: <one sentence from the passages, copied verbatim>\n"
#     "  NONE"
# )

# SPLIT_SYS = (
#     "A question cannot be answered from the retrieved passages, but a bridge sentence was found. "
#     "Split the question into EXACTLY TWO simpler questions using that bridge.\n"
#     "Rules:\n"
#     "- Question 1 must be answerable from the bridge sentence, and must be SIMPLER than the "
#     "original - never restate the original as question 1.\n"
#     "- Question 2 must contain the placeholder #1 (the answer to question 1) and, once #1 is "
#     "filled in, must answer the original question.\n"
#     "- Both must be about entities named in the question or the bridge sentence. Never mention an "
#     "entity from the examples.\n"
#     "- Output EXACTLY two lines, nothing else:\n"
#     "1. <question 1>\n"
#     "2. <question 2 containing #1>"
# )

# SPLIT_FEWSHOT = (
#     "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
#     "Question: In which city was the author of The Ashen Ledger born?\n"
#     "Bridge sentence: The Ashen Ledger was written by Ilvane Sabreth.\n"
#     "1. Who is the author of The Ashen Ledger?\n"
#     "2. In which city was #1 born?\n\n"
#     "Question: Which country is the spouse of Marren Volkov a citizen of?\n"
#     "Bridge sentence: Marren Volkov married Orrin Fennwick in 1998.\n"
#     "1. Who is Marren Volkov married to?\n"
#     "2. Which country is #1 a citizen of?\n\n"
#     "Question: Which time zone does Harrowmere use?\n"
#     "Bridge sentence: Harrowmere is a village in the province of Calvessa.\n"
#     "1. Which province is Harrowmere a village in?\n"
#     "2. Which time zone does #1 use?\n\n"
#     "Question: In which branch of the armed forces did Corvin Ashdale serve?\n"
#     "Bridge sentence: Corvin Ashdale held the rank of Vane Marshal.\n"
#     "1. Which rank did Corvin Ashdale hold?\n"
#     "2. Which branch of the armed forces uses the rank of #1?\n"
#     "### END EXAMPLES\n\n"
#     "Now the REAL question. Both questions must be about the entities in THIS question and its "
#     "bridge, and nothing from the examples above.\n"
# )

DECOMP_SYS = (
    "You turn a multi-hop CLAIM into the ordered list of questions someone would have to answer "
    "in order to check it.\n"
    "A multi-hop claim usually describes an entity instead of naming it - 'the comedian who "
    "founded Apatow Productions', 'the director of that film' - and its evidence is spread over "
    "several Wikipedia articles. The early questions resolve those descriptions into names; the "
    "later ones ask about the properties the claim asserts of them.\n"
    "Rules:\n"
    "- One question per line, numbered '1.', '2.', ...\n"
    "- A later question can refer to an earlier answer with #k (e.g. #1 = the answer to question 1).\n"
    "- Resolve every thing the claim describes but does not name, before asking about it.\n"
    "- Ask about EVERY property the claim asserts - dates, nationalities, roles, genres, places. "
    "A property nobody asks about is never checked, and a false claim is often false precisely "
    "there.\n"
    "- Ask, never assert. Nothing in the "
    "claim may be assumed true: that is what is being tested.\n"
    "- Each question must be answerable on its own.\n"
    "Output ONLY the numbered list, nothing else."
)

DECOMP_FEWSHOT = (
    "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
    "Claim: The Ashen Ledger, a 1998 mystery novel, was written by the Kerelian journalist who "
    "founded Northwind Cartography.\n"
    "1. Who founded Northwind Cartography?\n"
    "2. Who wrote The Ashen Ledger?\n"
    "3. What is the nationality of #1?\n"
    "4. What is the occupation of #1?\n"
    "5. In what year was The Ashen Ledger published?\n"
    "6. What genre is The Ashen Ledger?\n"
    "Claim: The lead actress of Glass Aviary was born in Dunmarrow.\n"
    "1. Who is the lead actress of Glass Aviary?\n"
    "2. Where was #1 born?\n\n"
    "Claim: Orrin Fennwick is known for his work in a type of stage play performed without spoken dialogue.\n"
    "1. What type of stage play is performed without spoken dialogue?\n"
    "2. What is Orrin Fennwick known for?\n"
    "### END EXAMPLES\n\n"
    "Now the REAL claim. Every question must be about the entities in THIS claim and nothing from "
    "the examples above.\n"
)

# Answers one hop from its passages. "unknown" is the refusal that fires a split, and on a
# 3-way benchmark it is also the honest answer for a hop the corpus does not cover.
ANSWER_SYS = (
    "You answer questions using ONLY the passages you are given. Answer the Question from the "
        "Passages. You may combine passages if the answer needs more than one of them. Your answer "
        "MUST be a word or phrase that appears in the passage text. If the passages truly do not "
        "contain the answer, output exactly: unknown. "
        "Never answer from your own knowledge of the world, even when you are confident and even as "
        "a fallback: an answer that is not supported by these passages is wrong here, whether or not "
        "it is true. "
        "Give a very concise answer - one word or short phrase only, with no explanation."
)

# Same job, for a node nothing could repair. It still prefers something over nothing, because the
# answer's only remaining use is as the next hop's search query - the verdict never sees it - but
# it may still decline when the passages are about something else entirely.
# On FC-MH this prompt is told never to refuse, because a wrong guess scores the same as a
# refusal there and a guess sometimes lands. Here a refusal is INFORMATION: a hop the corpus
# cannot answer is what NOT ENOUGH INFO means, and it is the class this benchmark loses on. So
# this prompt keeps the escape, and the count of hops that use it reaches the verdict below.
GUESS_SYS = ANSWER_SYS.replace(
    "If the passages truly do not contain the answer, output exactly: unknown. ",
    "Even when the passages are a poor match, give the closest word or phrase in them; never "
    "refuse and never answer 'unknown'. ")

# Reached only after ANSWER_SYS has said "unknown". Finds the entity to resolve first; never
# answers the question.
BRIDGE_SYS = (
    "You work ONLY over the passages you are given, never from your own knowledge of the world.\n"
    "A reader has already tried and failed to answer the Question from these passages. Do NOT "
    "answer the question. Your only job is to find the stepping stone.\n"
    "The passages often do not state the asked relation for the asked entity, but do state some "
    "OTHER relation of that entity, which names a NEW entity - and the asked relation is recorded "
    "for the new entity instead. They may not say which country a film was made in but may name "
    "its director; they may not give a person's birthplace but may name the band they founded; "
    "they may not say when a building opened but may name the company that built it.\n"
    "Find the ONE sentence in the passages that names that new entity, and copy it verbatim. It "
    "will normally mention an entity the question already names - that is how it connects. What "
    "makes it a stepping stone is the NEW name it introduces.\n"
    "A sentence is NOT a stepping stone if it introduces no new name, or if it is about an entity "
    "the question is not asking about.\n"
    "Output EXACTLY ONE line:\n"
    "  BRIDGE: <one sentence from the passages, copied verbatim>\n"
    "  NONE\n"
    "If no sentence in the passages names such an entity, output exactly: NONE"
)

# Turns (question, bridge sentence) into two subquestions, the second chaining on #1.
SPLIT_SYS = (
    "A question cannot be answered directly from the passages, but a bridge sentence was found. "
    "Split the question into EXACTLY TWO simpler subquestions using that bridge sentence.\n"
    "Rules:\n"
    "- Subquestion 1 must be answerable by the bridge sentence.\n"
    "- Subquestion 2 must contain the placeholder #1 (the answer of subquestion 1) and, once #1 "
    "is filled in, must answer the original question.\n"
    "- Both subquestions must be about entities named in the question or the bridge sentence you "
    "were given. Never mention an entity from the examples.\n"
    "- Output EXACTLY two lines, nothing else:\n"
    "1. <subquestion 1>\n"
    "2. <subquestion 2 containing #1>"
)

SPLIT_FEWSHOT = (
    "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
    "Question: In which city was the author of The Ashen Ledger born?\n"
    "Bridge sentence: The Ashen Ledger was written by Ilvane Sabreth.\n"
    "1. Who is the author of The Ashen Ledger?\n"
    "2. In which city was #1 born?\n\n"
    "Question: Which country is the spouse of Marren Volkov a citizen of?\n"
    "Bridge sentence: Marren Volkov married Orrin Fennwick in 1998.\n"
    "1. Who is Marren Volkov married to?\n"
    "2. Which country is #1 a citizen of?\n"
    "### END EXAMPLES\n\n"
    "Now the REAL question. Both subquestions must be about the entities in THIS question and its "
    "bridge sentence, and nothing from the examples above.\n"
)

print(f"prompts ready | max_split_depth={MAX_SPLIT_DEPTH} k={SOLVE_K} reader={READER_MODEL}")


In [ ]:
_local = threading.local()


class _ThreadLocalRetriever:
    def retrieve(self, query, k):
        active = getattr(_local, "retr", None)
        if active is None:
            raise RuntimeError("no retriever bound for this thread")
        out = []
        for passage, score in active.retrieve(query, k):
            idx = _local.index[id(passage)]
            title = (passage.get("title") or "").strip()
            body = (passage.get("text") or "").strip()
            out.append(({"serial": idx, "text": f"[{idx + 1}] {title}: {body}\n"}, score))
        return out


retriever = _ThreadLocalRetriever()


def _bind(active):
    _local.retr = active
    _local.index = {id(p): i for i, p in enumerate(active.passages)}


class _Response:
    def __init__(self, content):
        message = type("Message", (), {"content": content})()
        self.choices = [type("Choice", (), {"message": message})()]


class _Completions:
    @staticmethod
    def create(model=None, messages=None, max_tokens=256, **kwargs):
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        user = next((m["content"] for m in messages if m["role"] == "user"), "")
        return _Response(llm(system, user, max_tokens))


client = type("Client", (), {
    "chat": type("Chat", (), {"completions": _Completions()})()
})()
print("runtime ready")


In [ ]:
def decompose_question(claim):
    user = DECOMP_FEWSHOT + "\nClaim: " + " ".join(str(claim).split()) + "\n"
    response = client.chat.completions.create(
        model=READER_MODEL,
        messages=[{"role": "system", "content": DECOMP_SYS},
                  {"role": "user", "content": user}],
        max_tokens=256, temperature=0.0, seed=SEED)
    subquestions = []
    for line in (response.choices[0].message.content or "").splitlines():
        match = re.match(r"^\s*\d+[.)]\s*(.+)$", line.strip())
        if match:
            subquestions.append(match.group(1).strip())
    if not subquestions:
        subquestions = [str(claim)]
    return subquestions, len(subquestions)


def _fill(template, previous_answers):
    def replace(match):
        index = int(match.group(1)) - 1
        if not 0 <= index < len(previous_answers):
            return match.group(0)
        answer = str(previous_answers[index]).strip()
        return "" if answer.lower() == "unknown" else answer
    return " ".join(re.sub(r"#(\d+)", replace, template).split())


print("decomposer ready")


In [ ]:
import re, time


def _chat(system, user, max_tokens=96):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=READER_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                max_tokens=max_tokens, temperature=0.0, seed=42)
            return response.choices[0].message.content.strip()
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)


def _norm(value):
    return " ".join(re.sub(r"[^a-z0-9\s]", "", str(value).lower()).split())


def retrieve_pool(question):
    hits = retriever.retrieve(question, SOLVE_K)
    hits = sorted(hits, key=lambda hit: hit[0]["serial"])
    return "\n".join(fact["text"] for fact, _ in hits), hits


def answer_from_pool(question, pool):
    raw = _chat(ANSWER_SYS,
                f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:", 20)
    return raw.splitlines()[0].strip(), raw


def guess_from_pool(question, pool):
    raw = _chat(GUESS_SYS,
                f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:", 20)
    return raw.splitlines()[0].strip(), raw


def find_bridge(question, pool):
    raw = _chat(BRIDGE_SYS,
                f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nOutput:")
    line = next((line.strip() for line in raw.splitlines() if line.strip()), "")
    match = re.match(r"BRIDGE\s*:\s*(.+)$", line, re.I)
    if not match or _norm(match.group(1)) in ("", "none"):
        return None, raw
    return match.group(1).strip(), raw


def split(question, bridge):
    raw = _chat(SPLIT_SYS,
                SPLIT_FEWSHOT + f"\nQuestion: {question}\nBridge fact: {bridge}\n", 128)
    subquestions = []
    for line in raw.splitlines():
        match = re.match(r"^\s*[12][.)]\s*(.+)$", line.strip())
        if match:
            subquestions.append(match.group(1).strip())
    if (len(subquestions) < 2 or "#1" not in subquestions[1]
            or _norm(subquestions[0]) == _norm(question)):
        return None, raw
    return subquestions[:2], raw


def solve(question, depth, run):
    key = _norm(question)
    if key in run["memo"]:
        cached = run["memo"][key]
        run["trace"].append({"depth": depth, "question": question,
                             "pool": cached["pool"], "decision": "reused",
                             "answer": cached["answer"]})
        run["reused"] += 1
        return cached["answer"]

    pool, hits = retrieve_pool(question)
    node = {"depth": depth, "question": question,
            "pool": [{"serial": fact["serial"], "score": score, "text": fact["text"]}
                     for fact, score in hits]}
    run["trace"].append(node)
    run["max_depth"] = max(run["max_depth"], depth)

    def finish(decision, answer):
        node.update(decision=decision, answer=answer)
        run["memo"][key] = {"answer": answer, "pool": node["pool"]}
        return answer

    def commit(decision):
        answer, node["guess_raw"] = guess_from_pool(question, pool)
        run["calls"] += 1
        run["guesses"] += 1
        return finish(decision, answer)

    answer, node["answer_raw"] = answer_from_pool(question, pool)
    run["calls"] += 1
    if _norm(answer) != "unknown":
        return finish("answered", answer)

    if depth >= run["max_split_depth"]:
        return commit("committed (depth cap)")

    bridge, node["bridge_raw"] = find_bridge(question, pool)
    run["calls"] += 1
    if bridge is None:
        return commit("committed (no bridge)")

    pair, node["split_raw"] = split(question, bridge)
    run["calls"] += 1
    if pair is None:
        return commit("committed (split failed)")

    run["splits"] += 1
    node.update(decision="split", bridge=bridge, split=pair)
    node["answer"] = solve_sequence(pair, depth + 1, run)
    run["memo"][key] = {"answer": node["answer"], "pool": node["pool"]}
    return node["answer"]


def solve_sequence(subquestions, depth, run):
    answers = []
    for template in subquestions:
        answers.append(solve(_fill(template, answers), depth, run))
    return answers[-1] if answers else ""


def answer_question_recursive(claim, max_split_depth=MAX_SPLIT_DEPTH):
    subquestions, branches = decompose_question(claim)
    run = {"branches": branches, "calls": 1, "splits": 0, "guesses": 0,
           "reused": 0, "max_depth": 0, "max_split_depth": max_split_depth,
           "memo": {}, "trace": []}
    final = solve_sequence(subquestions, 0, run)
    run["branch_answers"] = [node["answer"] for node in run["trace"]
                             if node["depth"] == 0]
    run.pop("memo")
    return final, run


print("recursive pipeline ready")


In [ ]:
VERDICT_FEWSHOT = (
    "### EXAMPLES (format only - invented entities, do NOT reuse them)\n"
    "[Search record]\n"
    "Q: Who directed Glass Aviary?\nA: Ilvane Sabreth\n"
    "Q: What nationality is Ilvane Sabreth?\nA: Kerelian\n\n"
    "CLAIM: Glass Aviary was directed by a Kerelian director.\n"
    "Label: SUPPORT\n\n"
    "[Search record]\n"
    "Q: What kind of institution is Dunmarrow Museum?\nA: art museum\n"
    "Q: Is Dunmarrow Museum the largest museum in Kerelia?\nA: unknown\n\n"
    "CLAIM: Dunmarrow Museum is the largest museum in Kerelia.\n"
    "Label: NOT ENOUGH INFO\n\n"
    "[Search record]\n"
    "Q: How many regional titles has Quillon United won?\nA: unknown\n\n"
    "CLAIM: Quillon United has won 12 regional titles.\n"
    "Label: NOT ENOUGH INFO\n\n"
    "[Search record]\n"
    "Q: Where was Marren Volkov born?\nA: Ostravia\n\n"
    "CLAIM: Marren Volkov was born in Brenmark.\n"
    "Label: REFUTE\n"
    "### END EXAMPLES\n\n"
    "Now the REAL one. Judge only the supplied passages and/or search record below.\n\n"
)


VERDICT_SYS = (
    "You are a fact-checking verdict model. Decide how the evidence bears on the CLAIM.\n"
    "Reply with exactly one label and nothing else:\n"
    "- SUPPORT: every checkable part of the claim is stated by, or directly follows from, the "
    "evidence.\n"
    "- REFUTE: the evidence states something INCOMPATIBLE with the claim - a different value for "
    "something the claim asserts. Evidence that is merely silent about part of the claim is not "
    "a contradiction.\n"
    "- NOT ENOUGH INFO: the evidence neither states nor contradicts a fact the claim depends on.\n"
    "Three traps, in the order they cost the most:\n"
    "1. RANKING AND ORDER. Being the first, the second, the only, the youngest or the largest of "
    "something can only be settled by evidence that establishes the whole ordering. An article "
    "that names some of the items does not settle it. Where the evidence does not settle a "
    "ranking, the answer is NOT ENOUGH INFO, never REFUTE.\n"
    "2. A SPECIFIC QUANTITY. If the claim gives a count, an age, a year or a total and the "
    "evidence never states that number, the answer is NOT ENOUGH INFO, not SUPPORT. Evidence "
    "about the same subject is not evidence for the number.\n"
    "3. DESCRIPTION INSTEAD OF A NAME. A claim often refers to something by description rather "
    "than by name - 'the second-longest river in North America', 'the frontman of that band'. If "
    "the description fits something the evidence does name, treat them as the same thing. Not "
    "finding the description word-for-word is not a contradiction.\n"
    "Judge the substance of the claim. Paraphrase, different wording, and extra detail in the "
    "evidence do not matter, and an incidental modifier the evidence happens not to repeat is "
    "not by itself a reason to answer NOT ENOUGH INFO.\n"
    "Use ONLY the evidence provided. Your own world knowledge must not be used.\n"
    "Output the label only."
)


def parse_label(raw):
    text = " " + re.sub(r"[^a-z ]", " ", str(raw).lower()) + " "
    text = re.sub(r"\s+", " ", text)
    if re.search(r"not enough info|notenoughinfo|\bnei\b|insufficient|cannot be verified", text):
        return "NOT ENOUGH INFO"
    if re.search(r"\brefut|\bcontradict|\bfalse\b|\bdisproved?\b", text):
        return "REFUTE"
    if re.search(r"\bsupport|\bentail|\btrue\b|\bconfirmed\b", text):
        return "SUPPORT"
    return None


def format_pool(passages):
    return "\n\n".join(f"[{i + 1}] {p['title']}: {p['text']}"
                         for i, p in enumerate(passages))


def resolution_notes(steps):
    lines = [f"- {question} -> {answer}" for question, answer in (steps or [])
             if str(answer).strip().lower() != "unknown"]
    if not lines:
        return ""
    return "\n\n[Working notes]\n" + "\n".join(lines)


def verdict(claim, passages, steps=None):
    user = (f"{VERDICT_FEWSHOT}[Evidence]\n{format_pool(passages[:EVIDENCE_CAP])}"
            f"{resolution_notes(steps)}\n\nCLAIM: {claim}\nLabel:")
    return llm(VERDICT_SYS, user, max_tokens=16)


## 12. The arms

In [ ]:
class RecordingRetriever:
    def __init__(self, inner):
        self.inner = inner
        self.passages = inner.passages
        self.seen = {}

    def retrieve(self, query, k):
        results = self.inner.retrieve(query, k)
        for passage, _ in results:
            self.seen.setdefault(id(passage), passage)
        return results

    def collected(self):
        return list(self.seen.values())


def arm_retrieve(instance, recorder):
    passages = [p for p, _ in recorder.retrieve(instance["claim"], TOP_K)]
    raw = verdict(instance["claim"], passages)
    return raw, {"llm_calls": 1, "trace": [], "splits": 0, "guesses": 0,
                 "unresolved": 0, "n_top": 0}


def _run_recursive(instance, recorder, depth):
    _bind(recorder)
    claim_passages = [passage for passage, _ in recorder.retrieve(instance["claim"], TOP_K)]
    _, run = answer_question_recursive(instance["claim"], depth)

    top = [node for node in run["trace"] if node["depth"] == 0]
    unresolved = sum(_norm(node.get("answer", "")) == "unknown" for node in top)
    steps = [(node["question"], node.get("answer")) for node in top
             if node.get("decision") == "answered"]
    raw = verdict(instance["claim"], claim_passages, steps)
    return raw, {"llm_calls": run["calls"] + 1, "trace": run["trace"],
                 "splits": run["splits"], "guesses": run["guesses"],
                 "unresolved": unresolved, "n_top": len(top),
                 "reused": run.get("reused", 0)}


def arm_decompose(instance, recorder):
    return _run_recursive(instance, recorder, 0)


def arm_split(instance, recorder):
    return _run_recursive(instance, recorder, MAX_SPLIT_DEPTH)


ARM_FN = {"retrieve": arm_retrieve, "decompose": arm_decompose, "split": arm_split}
print("arms ready:", list(ARM_FN))


## 13. Run

In [ ]:
# ---------------- Run one arm over the sample ----------------

def chain_steps(trace):
    """(question, answer) per solved node - for reading traces only. The verdict never sees it."""
    out = []
    for t in trace or []:
        q, a = t.get("question", ""), t.get("answer")
        if q:
            out.append((str(q), "(unresolved)" if a in (None, "") else str(a)))
    return out


def run_arm(name, claims, base_retriever, workers=WORKERS):
    arm = ARM_FN[name]
    rows, traces = [], []
    t0 = time.time()
    _calls["n"] = 0
    _errs, _err_lock = [], threading.Lock()

    def one(inst):
        rec = RecordingRetriever(base_retriever)
        try:
            raw, info = arm(inst, rec)
        except QuotaExhausted:
            raise
        except (NameError, AttributeError, ImportError) as e:
            # A coding error, not a run-time one: almost always a cell that was never executed.
            # Scoring it as a wrong answer would make a whole arm read as a bad result.
            raise RuntimeError(
                f"{name}: {type(e).__name__}: {e}" + chr(10) +
                "  This is a missing NAME, not an API failure -- run every cell above, in order. "
                "This arm is not being scored.") from e
        except Exception as e:
            raw, info = f"(error: {e})", {"llm_calls": 0, "trace": [], "splits": 0,
                                          "guesses": 0, "unresolved": 0, "n_top": 0}
            with _err_lock:
                _errs.append(f"{type(e).__name__}: {e}")
                if len(_errs) == 1:
                    print(f"  !! {name}: first error -- {type(e).__name__}: {str(e)[:150]}")
        steps = chain_steps(info.get("trace"))
        traces.append({"id": inst["id"], "claim": inst["claim"], "gold": inst["label"],
                       "pred": parse_label(raw) or "UNPARSED",
                       "steps": [{"q": q, "a": a} for q, a in steps]})
        return {"id": inst["id"], "pred": parse_label(raw) or "UNPARSED", "gold": inst["label"],
                "raw": str(raw)[:200], "n_hops": inst.get("n_hops", 0), "n_subq": len(steps),
                "n_splits": info.get("splits", 0), "n_guesses": info.get("guesses", 0),
                "n_unresolved": info.get("unresolved", 0), "n_top": info.get("n_top", 0),
                "llm_calls": info.get("llm_calls", 0), "n_passages": len(rec.collected())}

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(one, c): c for c in claims}
        for f in tqdm(as_completed(futs), total=len(futs), desc=name):
            rows.append(f.result())

    df = pd.DataFrame(rows).set_index("id").loc[[c["id"] for c in claims]]
    print(f"{name}: {len(df)} claims, {_calls['n']} calls, {time.time()-t0:.0f}s"
          + (f"   !! {len(_errs)}/{len(df)} ERRORED" if _errs else ""))
    if _errs:
        print("   an arm with error rows is NOT comparable: those claims score as wrong. "
              f"Distinct: {sorted(set(_errs))[:3]}")
    return df, traces


print("runner ready")

In [ ]:
FRAMES, TRACES = {}, {}

In [ ]:
ARMS  = ["retrieve", "decompose", "split"]
for _arm in ARMS:
    FRAMES[_arm], TRACES[_arm] = run_arm(_arm, CLAIMS, base_retriever)


In [ ]:
LABELS = ["SUPPORT", "REFUTE", "NOT ENOUGH INFO"]


def macro_f1(frame):
    scores = []
    for label in LABELS:
        tp = ((frame["pred"] == label) & (frame["gold"] == label)).sum()
        fp = ((frame["pred"] == label) & (frame["gold"] != label)).sum()
        fn = ((frame["pred"] != label) & (frame["gold"] == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall)
                      if precision + recall else 0.0)
    return 100 * sum(scores) / len(scores)


def results_table(frames):
    print(f"{'arm':<12}{'acc':>7}{'macroF1':>9}{'calls/q':>9}{'passages':>10}{'split%':>8}")
    for name, frame in frames.items():
        accuracy = 100 * (frame["pred"] == frame["gold"]).mean()
        print(f"{name:<12}{accuracy:>7.1f}{macro_f1(frame):>9.1f}"
              f"{frame['llm_calls'].mean():>9.1f}{frame['n_passages'].mean():>10.1f}"
              f"{100 * (frame['n_splits'] > 0).mean():>7.0f}%")

        print("class recall:", end=" ")
        values = []
        for label in LABELS:
            selected = frame[frame["gold"] == label]
            values.append(f"{label}={100 * (selected['pred'] == label).mean():.1f}%")
        print(", ".join(values))

    unparsed = {name: int((frame["pred"] == "UNPARSED").sum())
                for name, frame in frames.items()}
    if any(unparsed.values()):
        print("UNPARSED rows:", unparsed)


In [ ]:
results_table(FRAMES)


In [ ]:
out = os.path.join(
    HERE, f"nb_rec_exfever_{RETRIEVER_NAME}_{READER_MODEL}_n{len(CLAIMS)}_full_run_2.json")
json.dump({
    "config": {"retriever": RETRIEVER_NAME, "reader": READER_MODEL,
               "labels": "3way", "top_k": TOP_K, "solve_k": SOLVE_K,
               "evidence_cap": EVIDENCE_CAP, "n": len(CLAIMS),
               "max_split_depth": MAX_SPLIT_DEPTH, "seed": SEED,
               "verdict_evidence": "passages+notes", "seed_with_claim": True},
    "frames": {name: frame.reset_index().to_dict("records")
               for name, frame in FRAMES.items()},
    "traces": TRACES
}, io.open(out, "w", encoding="utf-8"), indent=1, ensure_ascii=False, default=str)
print("saved:", out)
